In [ ]:
# Cell 1: Install dependencies
!pip install pydub beautifulsoup4 requests
!apt-get -qq install -y ffmpeg

In [ ]:
# Cell 2: Clone repo and set configuration
# IMPORTANT: Push your local commits BEFORE running this cell!
!git clone https://github.com/bibleman-stan/readers-bofm.git
%cd readers-bofm

# ══════════════════════════════════════════
# CONFIGURATION — Edit these for each run
# ══════════════════════════════════════════
ELEVENLABS_API_KEY = "YOUR_ELEVENLABS_API_KEY_HERE"

# Voice: Sister M
VOICE_ID = "iJpuSZDxZSBJHFSeavba"
VOICE_NAME = "sister-m"           # used in filenames
MODEL_ID = "eleven_multilingual_v2"

# Book to generate — CHANGE THESE PER RUN
BOOK_ID = '2nephi'               # e.g. '2nephi', 'mosiah', 'helaman'
BOOK_NAME = '2 Nephi'            # human-readable
TOTAL_CHAPTERS = 33              # chapter count

LINE_PAUSE_MS = 180
VERSE_PAUSE_MS = 500

print(f'Config set!')
print(f'  Voice: {VOICE_NAME} ({VOICE_ID})')
print(f'  Book: {BOOK_NAME} ({BOOK_ID}), {TOTAL_CHAPTERS} chapters')
print(f'  Model: {MODEL_ID}')

In [ ]:
# Cell 3: Core functions — HTML parsing + ElevenLabs TTS + Line Caching
import re, json, time, hashlib, requests, io, os
from pathlib import Path
from bs4 import BeautifulSoup, NavigableString
from pydub import AudioSegment

VOICE_SETTINGS = {
    'stability': 0.50,
    'similarity_boost': 0.75,
    'style': 0.35,
    'use_speaker_boost': True,
}

CACHE_DIR = Path('audio_cache')
CACHE_DIR.mkdir(exist_ok=True)

def cache_key(text, voice_id, model_id, settings):
    blob = json.dumps({'text': text, 'voice_id': voice_id, 'model_id': model_id, 'settings': settings}, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]

def get_cached_audio(key):
    path = CACHE_DIR / f'{key}.mp3'
    return AudioSegment.from_mp3(path) if path.exists() else None

def save_to_cache(key, audio_seg):
    audio_seg.export(str(CACHE_DIR / f'{key}.mp3'), format='mp3', bitrate='128k')

def get_text_from_element(el, use_modern=False):
    clone = BeautifulSoup(str(el), 'html.parser')
    if use_modern:
        for swap in clone.find_all(class_=['swap', 'swap-quiet']):
            mod = swap.get('data-mod')
            if mod: swap.string = mod
    for remove in clone.find_all(class_=['verse-num', 'parry-label-spacer', 'parry-label']):
        remove.decompose()
    return re.sub(r'\s+', ' ', clone.get_text()).strip()

def parse_chapter(html_path, book_id, chapter_num):
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    chapter_div = soup.find(id=f'ch-{book_id}-{chapter_num}')
    if not chapter_div:
        print(f'  Warning: ch-{book_id}-{chapter_num} not found')
        return []
    items, line_idx = [], 0
    for child in chapter_div.children:
        if isinstance(child, NavigableString): continue
        classes = child.get('class', [])
        if 'pericope-header' in classes:
            line_idx += 1
            continue
        if 'verse' in classes:
            vn_el = child.find(class_='verse-num')
            vn = vn_el.get_text().strip() if vn_el else ''
            for line_el in child.find_all('span', recursive=False):
                if 'line' not in (line_el.get('class') or []): continue
                text = get_text_from_element(line_el, use_modern=False)
                if text:
                    items.append({'type': 'line', 'text': text, 'verse_num': vn, 'line_index': line_idx})
                    line_idx += 1
            if items and items[-1]['type'] != 'verse-gap':
                items.append({'type': 'verse-gap', 'text': '', 'verse_num': '', 'line_index': -1})
    return items

def tts_elevenlabs(text, voice_id=None, model_id=None, settings=None):
    voice_id = voice_id or VOICE_ID
    model_id = model_id or MODEL_ID
    settings = settings or VOICE_SETTINGS
    key = cache_key(text, voice_id, model_id, settings)
    cached = get_cached_audio(key)
    if cached is not None: return cached, True
    url = f'https://api.elevenlabs.io/v1/text-to-speech/{voice_id}'
    headers = {'xi-api-key': ELEVENLABS_API_KEY, 'Content-Type': 'application/json', 'Accept': 'audio/mpeg'}
    payload = {'text': text, 'model_id': model_id, 'voice_settings': settings}
    resp = requests.post(url, json=payload, headers=headers)
    if resp.status_code != 200:
        print(f'  API error {resp.status_code}: {resp.text[:200]}')
        return AudioSegment.silent(duration=500), False
    audio_seg = AudioSegment.from_mp3(io.BytesIO(resp.content))
    save_to_cache(key, audio_seg)
    return audio_seg, False

def stitch_chapter(items, voice_id=None, model_id=None, settings=None,
                   line_pause_ms=None, verse_pause_ms=None, stitch_only=False):
    voice_id = voice_id or VOICE_ID
    model_id = model_id or MODEL_ID
    settings = settings or VOICE_SETTINGS
    line_pause_ms = line_pause_ms or LINE_PAUSE_MS
    verse_pause_ms = verse_pause_ms or VERSE_PAUSE_MS
    combined = AudioSegment.empty()
    timing, current_time_ms = [], 0
    stats = {'cached': 0, 'generated': 0, 'skipped': 0, 'chars_used': 0}
    speakable = [i for i in items if i['type'] == 'line']
    total = len(speakable)
    done = 0
    for item in items:
        if item['type'] == 'verse-gap':
            combined += AudioSegment.silent(duration=verse_pause_ms)
            current_time_ms += verse_pause_ms
            continue
        if item['type'] != 'line': continue
        done += 1
        key = cache_key(item['text'], voice_id, model_id, settings)
        if stitch_only:
            cached = get_cached_audio(key)
            if cached is None:
                stats['skipped'] += 1; continue
            audio_seg, tag = cached, 'cache'
            stats['cached'] += 1
        else:
            audio_seg, from_cache = tts_elevenlabs(item['text'], voice_id, model_id, settings)
            if from_cache:
                stats['cached'] += 1; tag = 'cache'
            else:
                stats['generated'] += 1
                stats['chars_used'] += len(item['text'])
                tag = 'NEW'
                time.sleep(0.05)
        start_ms = current_time_ms
        combined += audio_seg
        current_time_ms += len(audio_seg)
        timing.append({'start': round(start_ms/1000,3), 'end': round(current_time_ms/1000,3),
                       'type': 'line', 'text': item['text'], 'verse': item['verse_num'],
                       'lineIndex': item['line_index']})
        print(f'  [{done}/{total}] [{tag}] v{item["verse_num"]}: {item["text"][:50]}...')
        combined += AudioSegment.silent(duration=line_pause_ms)
        current_time_ms += line_pause_ms
    return combined, timing, stats

print('Core functions loaded!')
print(f'  Cache dir: {CACHE_DIR}')

In [ ]:
# Cell 4: DRY RUN — Count characters before spending credits
# Run this FIRST to see exactly how many chars/credits you'll use.

total_lines = 0
total_chars = 0

for ch in range(1, TOTAL_CHAPTERS + 1):
    items = parse_chapter(f'books/{BOOK_ID}.html', BOOK_ID, ch)
    speakable = [i for i in items if i['type'] == 'line']
    chars = sum(len(i['text']) for i in speakable)
    total_lines += len(speakable)
    total_chars += chars
    print(f'  Ch {ch:2d}: {len(speakable):3d} lines, {chars:5d} chars')

print(f'\n{"="*50}')
print(f'{BOOK_NAME}: {TOTAL_CHAPTERS} chapters')
print(f'  Total lines: {total_lines}')
print(f'  Total chars: {total_chars:,}')
print(f'  Voice: {VOICE_NAME}')
print(f'{"="*50}')
print(f'\nIf this looks right, proceed to Cell 5 (voice test) or Cell 6 (full gen).')

In [ ]:
# Cell 5: VOICE TEST — 5 lines from Chapter 1 (~250 chars)
# Listen before committing to full generation!
from IPython.display import Audio, display

items = parse_chapter(f'books/{BOOK_ID}.html', BOOK_ID, 1)
speakable = [i for i in items if i['type'] == 'line']
test_items = speakable[:5]

test_chars = sum(len(i['text']) for i in test_items)
print(f'Voice test ({VOICE_NAME}): 5 lines from {BOOK_NAME} 1')
print(f'Chars: ~{test_chars}\n')

combined_test = AudioSegment.empty()
for i, item in enumerate(test_items):
    print(f'  [{i+1}/5] {item["text"][:60]}...')
    seg, cached = tts_elevenlabs(item['text'])
    print(f'         [{"cache" if cached else "NEW"}]')
    combined_test += seg + AudioSegment.silent(duration=LINE_PAUSE_MS)
    if not cached: time.sleep(0.1)

combined_test.export('test_voice.mp3', format='mp3', bitrate='128k')
print(f'\n{VOICE_NAME} test: {len(combined_test)/1000:.1f}s')
display(Audio('test_voice.mp3'))

In [ ]:
# Cell 6: FULL GENERATION — All chapters
# Per-line caching: re-running is cheap (cached lines skip API)

Path('audio').mkdir(exist_ok=True)
results = []

for ch in range(1, TOTAL_CHAPTERS + 1):
    print(f'\n{"="*60}')
    print(f'CHAPTER {ch} of {TOTAL_CHAPTERS}')
    print(f'{"="*60}')

    items = parse_chapter(f'books/{BOOK_ID}.html', BOOK_ID, ch)
    speakable = [i for i in items if i['type'] == 'line']
    ch_chars = sum(len(i['text']) for i in speakable)
    print(f'  {len(speakable)} lines, ~{ch_chars} chars')

    combined, timing, stats = stitch_chapter(items)

    mp3_path = f'audio/{BOOK_ID}-{ch}-{VOICE_NAME}.mp3'
    combined.export(mp3_path, format='mp3', bitrate='128k')
    duration_s = len(combined) / 1000
    file_size = os.path.getsize(mp3_path)

    json_path = f'audio/{BOOK_ID}-{ch}-{VOICE_NAME}.json'
    manifest = {
        'book': BOOK_ID, 'bookName': BOOK_NAME, 'chapter': ch,
        'voice': {'provider': 'elevenlabs', 'voice_id': VOICE_ID,
                  'model_id': MODEL_ID, 'settings': VOICE_SETTINGS,
                  'name': VOICE_NAME},
        'pauses': {'line_ms': LINE_PAUSE_MS, 'verse_ms': VERSE_PAUSE_MS},
        'duration': round(duration_s, 3), 'lines': timing,
    }
    with open(json_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    results.append({'ch': ch, 'lines': len(speakable), 'duration': duration_s,
                    'size_kb': file_size/1024, 'cached': stats['cached'],
                    'generated': stats['generated'], 'chars_used': stats['chars_used']})
    print(f'  MP3: {duration_s:.1f}s, {file_size/1024:.0f}KB')
    print(f'  Cache: {stats["cached"]}, New: {stats["generated"]}, Credits: ~{stats["chars_used"]}')

# Summary
print(f'\n\n{"="*60}')
print(f'{BOOK_NAME} COMPLETE — {TOTAL_CHAPTERS} chapters with {VOICE_NAME}')
print(f'{"="*60}')
td = sum(r['duration'] for r in results)
ts = sum(r['size_kb'] for r in results)
tl = sum(r['lines'] for r in results)
tn = sum(r['generated'] for r in results)
tc = sum(r['cached'] for r in results)
tu = sum(r['chars_used'] for r in results)
print(f'  Total lines: {tl}')
print(f'  Total duration: {td/60:.1f} min ({td:.0f}s)')
print(f'  Total size: {ts/1024:.1f} MB')
print(f'  Cache hits: {tc}, Newly generated: {tn}')
print(f'  Credits used: ~{tu:,} chars')
for r in results:
    print(f'  Ch{r["ch"]:2d}: {r["lines"]:3d} lines, {r["duration"]:6.1f}s, {r["size_kb"]:6.0f}KB')

In [ ]:
# Cell 7: Download as zip
import zipfile
from google.colab import files

zip_path = f'audio/{BOOK_ID}_{VOICE_NAME}_audio.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for ch in range(1, TOTAL_CHAPTERS + 1):
        mp3 = f'audio/{BOOK_ID}-{ch}-{VOICE_NAME}.mp3'
        jsn = f'audio/{BOOK_ID}-{ch}-{VOICE_NAME}.json'
        if os.path.exists(mp3): zf.write(mp3, os.path.basename(mp3))
        if os.path.exists(jsn): zf.write(jsn, os.path.basename(jsn))

zip_size = os.path.getsize(zip_path) / (1024*1024)
print(f'Zip: {zip_path} ({zip_size:.1f} MB)')
print('Downloading...')
files.download(zip_path)